<a href="https://colab.research.google.com/github/apcyssr/2026_Spring_KBU/blob/main/20260327_APICHAYA_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install gradio pandas requests

In [15]:
import requests
import pandas as pd

# ดึงข้อมูล
url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url)
df = pd.DataFrame(response.json())

# แปลงคอลัมน์ address (Series) ให้เป็น DataFrame แยกต่างหาก
address_df = pd.json_normalize(df['address'])

print("--- Address DataFrame ---")
display(address_df.head())

--- Address DataFrame ---


,street,suite,city,zipcode,geo.lat,geo.lng
0,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496
1,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618
2,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653
3,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990
4,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342


In [16]:
url_posts = "https://jsonplaceholder.typicode.com/posts"
df_posts = pd.read_json(url_posts)

# Filter for even indices only
df_even = df_posts.iloc[::2]

# Save to Excel (Make sure 'openpyxl' is installed)
file_name = "even_posts.xlsx"
df_even.to_excel(file_name, index=False)

# เปลี่ยนบรรทัดสุดท้ายเป็นภาษาอังกฤษ
print(f"--- Task 6 Result: Successfully saved '{file_name}' with {len(df_even)} rows ---")

--- Task 6 Result: Successfully saved 'even_posts.xlsx' with 50 rows ---


In [18]:
import requests
import pandas as pd
import gradio as gr

def filter_posts(user_id):
    url = "https://jsonplaceholder.typicode.com/posts"
    # API 파라미터를 사용하여 데이터 필터링 (ใช้ params ในการกรองข้อมูล)
    params = {'userId': user_id}
    res = requests.get(url, params=params)

    filter_df = pd.DataFrame(res.json())
    return filter_df

# Gradio UI 구성 (สร้างหน้าจอ UI เป็นภาษาเกาหลี)
demo = gr.Interface(
    fn=filter_posts,
    inputs=gr.Number(label="User ID 입력 (1-10)"), # เปลี่ยนเป็น "กรอก User ID"
    outputs=gr.DataFrame(label="필터링된 결과"),      # เปลี่ยนเป็น "ผลลัพธ์ที่กรองแล้ว"
    title="User ID별 포스트 데이터 필터링 시스템",       # เปลี่ยนเป็น "ระบบกรองข้อมูล Post ตาม User ID"
    description="User ID를 입력하면 해당 사용자의 데이터를 실시간으로 보여줍니다." # เพิ่มคำอธิบายการใช้งาน
)

# 실행 (เปิดใช้งาน)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://87d43ee992f29d58fc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
from google.colab import files
uploaded = files.upload() # พอกดรันจะมีปุ่มให้กดเลือกไฟล์จากเครื่องครับ

Saving 서울특별시 공공자전거 이용정보(일별)_2510.csv to 서울특별시 공공자전거 이용정보(일별)_2510.csv
Saving 서울특별시 공공자전거 이용정보(일별)_2511.csv to 서울특별시 공공자전거 이용정보(일별)_2511.csv
Saving 서울특별시 공공자전거 이용정보(일별)_2512.csv to 서울특별시 공공자전거 이용정보(일별)_2512.csv


In [20]:
import pandas as pd
import gradio as gr
import requests

# 1. ข้อมูลดาวน์โหลดและรวมไฟล์ (데이터 로드 및 병합)
def load_and_merge():
    file_list = [
        '서울특별시 공공자전거 이용정보(일별)_2510.csv',
        '서울특별시 공공자전거 이용정보(일별)_2511.csv',
        '서울특별시 공공자전거 이용정보(일별)_2512.csv'
    ]

    list_df = []
    for file in file_list:
        try:
            # ใช้ encoding='cp949' สำหรับภาษาเกาหลี
            temp_df = pd.read_csv(file, encoding='cp949')
            list_df.append(temp_df)
            print(f"파일 로드 완료: {file}")
        except Exception as e:
            print(f"로드 오류 {file}: {e}")

    if not list_df:
        return pd.DataFrame()

    full_df = pd.concat(list_df, ignore_index=True)
    return full_df

# โหลดข้อมูล (데이터 미리 로드)
df_bike = load_and_merge()

# 2. ฟังก์ชันวิเคราะห์ (분석 함수)
def get_summary():
    return df_bike.describe()

def get_top_stations(n=10):
    col_name = '대여소명' if '대여소명' in df_bike.columns else df_bike.columns[1]
    # เปลี่ยนชื่อคอลัมน์ในผลลัพธ์ให้เป็นภาษาเกาหลี
    result = df_bike[col_name].value_counts().head(n).reset_index()
    result.columns = ['대여소 이름', '이용 횟수']
    return result

# 3. สร้าง Gradio UI (대시보드 구성)
with gr.Blocks() as demo:
    gr.Markdown("# 🚲 서울특별시 공공자전거 이용 현황 대시보드 (2025년 4분기)")

    with gr.Tabs():
        # หน้าแรก: ข้อมูลดิบ
        with gr.TabItem("전체 데이터 보기"):
            gr.DataFrame(df_bike.head(100), label="상위 100개 데이터 샘플")

        # หน้าสอง: สถิติ
        with gr.TabItem("통계 요약"):
            btn_stat = gr.Button("통계 데이터 생성")
            stat_output = gr.DataFrame(label="기본 통계 분석 결과")
            btn_stat.click(get_summary, outputs=stat_output)

        # หน้าสาม: อันดับสถานี
        with gr.TabItem("인기 대여소 순위"):
            num_input = gr.Slider(minimum=5, maximum=30, step=5, label="순위 개수 설정", value=10)
            btn_rank = gr.Button("순위 보기")
            rank_output = gr.DataFrame(label="대여소별 이용 순위")
            btn_rank.click(get_top_stations, inputs=num_input, outputs=rank_output)

# 4. รันโปรแกรม (실행)
if __name__ == "__main__":
    demo.launch(share=True)

파일 로드 완료: 서울특별시 공공자전거 이용정보(일별)_2510.csv
파일 로드 완료: 서울특별시 공공자전거 이용정보(일별)_2511.csv
파일 로드 완료: 서울특별시 공공자전거 이용정보(일별)_2512.csv
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://969e26c6c0cae5ef36.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
import requests
import pandas as pd
import gradio as gr
from google.colab import ai  # Google Colab의 AI 에이전트 호출

def translate_agent(user_id):
    # 1. API를 통해 데이터 가져오기 (7번 과제 방식)
    url = "https://jsonplaceholder.typicode.com/posts"
    params = {'userId': user_id}
    res = requests.get(url, params=params)
    df = pd.DataFrame(res.json())

    if df.empty:
        return "데이터를 찾을 수 없습니다.", "데이터를 찾을 수 없습니다."

    # 2. 첫 번째 행(Index 0)ของ body 컬럼 데이터 추출
    first_body = df.iloc[0]['body']

    # 3. Colab AI를 사용하여 번역 수행 (API 키 없이 사용)
    # 영어 본문을 한국어로 번역하도록 프롬프트 설정
    prompt = f"Translate the following English text to Korean: {first_body}"
    translated_text = ai.generate_text(prompt)

    return first_body, translated_text

# Gradio UI 구성 (UI를 한국어로 변경)
demo_translate = gr.Interface(
    fn=translate_agent,
    inputs=gr.Number(label="User ID 입력 (1-10)"),
    outputs=[
        gr.Textbox(label="원본 영문 텍스트 (첫 번째 행)"),
        gr.Textbox(label="AI 한국어 번역 결과")
    ],
    title="Gemini AI 번역기 (Colab 에이전트)",
    description="User ID를 입력하면 해당 사용자의 첫 번째 포스트 내용을 AI가 실시간으로 번역해줍니다."
)

# 실행
if __name__ == "__main__":
    demo_translate.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://902dcf8a32ec1fd89a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
